# engine

> nada (wav2vec 2.0 on Modular MAX), loaded straight into this process.

In [ ]:
#| default_exp engine

In [ ]:
#| hide
from nbdev.showdoc import *

An embedding is the mean final hidden state over the clip, minus a fixed "typical audio" vector (`models/embed_center.npy`), scaled to unit length. Clips are padded up to a whole second because each length is its own compiled graph (a cold compile takes about a minute). MAX is only touched from one worker thread, so loading and every embed run there.

In [ ]:
#| export
import asyncio, base64, os, sys, threading, time, traceback
import numpy as np
from collections import OrderedDict
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

In [ ]:
#| export
os.environ.setdefault('MODULAR_HOME', f'{sys.prefix}/share/max')  # what pixi's activation sets, in case we weren't started by pixi
from nadayantra.core import repo_root
root = repo_root()
models = Path(os.environ.get('NADA_MODELS') or root.parent/'nada'/'models')
sr, maxsec, dim = 16_000, 10, 768

In [ ]:
#| export
def pcm(b64):
    "16-bit little-endian PCM from the browser, as float32 in -1..1"
    raw = base64.b64decode(b64)
    raw = raw[:len(raw)//2*2]
    return np.frombuffer(raw, '<i2').astype('f4')[:sr*maxsec]/32767 if raw else None

In [ ]:
#| export
def trim(x):
    "Cut the quiet lead-in and tail so a sound embeds the same however long you waited"
    if x is None or not x.size or (peak := np.abs(x).max()) < .01: return None
    loud = np.flatnonzero(np.abs(x) >= max(peak*.15, .01))
    pad, n = int(sr*.1), x.size-1
    a, b = max(loud[0]-pad, 0), min(loud[-1]+pad, n)
    short = int(sr*.5) - (b-a+1)  # a click gets a little more room
    if short > 0: a,b = max(a-short//2, 0), min(b+short-short//2, n)
    return x[a:b+1]

In [ ]:
#| export
class Engine:
    def __init__(self):
        from nada.config import base_960h_config
        from nada.model import build, load_checkpoint
        self.cfg, self.device, self.graphs = base_960h_config(), 'cpu', OrderedDict()
        self.center = np.load(root/'models'/'embed_center.npy').astype('f4').reshape(-1)
        t = time.time()
        self.model = build(self.cfg)
        load_checkpoint(self.model, models/'wav2vec2.safetensors')
        print(f'[nada] loaded weights in {time.time()-t:.1f}s')
        self.graph(2*sr)  # a typical length, so a cold cache costs the splash and not the first sound

    def graph(self, n):
        from nada.model import compile_model
        g = self.graphs.pop(n, None)
        if g is None:
            t = time.time()
            g = compile_model(self.model, batch=1, num_samples=n)
            print(f'[nada] compiled {n//sr}s graph in {time.time()-t:.1f}s')
        self.graphs[n] = g
        while len(self.graphs) > 4: self.graphs.popitem(last=False)
        return g

    def embed(self, x):
        from nada.features import normalize
        from nada.model import to_device
        x = x[:sr*maxsec]
        if x.size < sr//10: raise ValueError('That sound was too short. Hold it a little longer.')
        real = self.cfg.output_length(x.size)
        x = np.pad(normalize(x), (0, -x.size % sr))
        _,h = self.graph(x.size)(to_device(self.model, x[None]))
        v = np.asarray(h.to_numpy(), dtype='f4')[0,:real].mean(0) - self.center
        return v/max(float(np.linalg.norm(v)), 1e-12)

In [ ]:
#| export
class Nada:
    "One shared model with a state the UI can poll: idle, starting, ready or failed"
    def __init__(self): self.state,self.device,self.error,self.eng = 'idle',None,None,None; self.lock,self.pool = threading.Lock(), ThreadPoolExecutor(1)

    def warm_up(self):
        with self.lock:
            if self.state in ('starting','ready'): return
            self.state,self.error = 'starting',None
        self.pool.submit(self._load)

    def _load(self):
        try: self.eng = Engine(); self.device,self.state = self.eng.device,'ready'
        except Exception as e: traceback.print_exc(); self.error,self.state = f'{type(e).__name__}: {e}','failed'

    def _embed(self, x):
        if not self.eng: raise RuntimeError(f'nada could not start. {self.error or ""}')
        return self.eng.embed(x)

    async def embed(self, x):
        if self.state == 'idle': self.warm_up()
        return await asyncio.get_running_loop().run_in_executor(self.pool, self._embed, x)  # queues behind loading

In [ ]:
#| export
nada = Nada()